# Customer Transaction & Behavior Analysis
**DSA1080A – Programming for Data Science | Spring Semester 2026 | USIU-Africa**

**Student:** Gad Rubuye | **Student ID:** 677569


## Section 1 — Setup & Imports

In [2]:
# Importing all libraries required across the full project.
# Grouped by purpose for clarity and maintainability.

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Display settings — ensures plots render inline in Jupyter
%matplotlib inline

# Setting a clean, consistent visual style for all charts
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Suppressing non-critical warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

All libraries imported successfully.


---
## Section 2 — Data Loading & First Inspection

In [5]:
# Loading the raw dataset using ISO-8859-1 encoding.
# This encoding is required because the file contains special characters
# (e.g. accented letters in product descriptions) that UTF-8 cannot parse.
# Without specifying this, pandas raises a UnicodeDecodeError.

df = pd.read_csv('/Users/mac/Desktop/DSA1080/Final Project/data/raw_dataset.csv', encoding='ISO-8859-1')

print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Dataset loaded successfully.
Shape: 541,909 rows × 8 columns


In [6]:
# Inspecting column names, data types, and non-null counts.
# This gives us a complete structural overview before any cleaning begins.
# Key things to look for: object types that should be dates or numbers,
# and columns with fewer non-null entries than the total row count.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [7]:
# Generating descriptive statistics for all numeric columns.
# We use the median alongside the mean to detect skewness early.
# If mean >> median, the distribution is right-skewed — common in revenue data
# where a small number of high-value transactions pull the average upward.

df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [8]:
# Previewing the first 5 rows to verify structure and content.
# This confirms column values match expected formats before we begin cleaning.

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


---
## Section 3 — Data Cleaning

### 3.1 — Missing Value Analysis

In [9]:
# Calculating missing value count and percentage for every column.
# Percentage is more meaningful than count alone — it tells us
# whether missing data is isolated noise or a structural problem.

missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct.round(2)
})

# Only showing columns that actually have missing values
print(missing_summary[missing_summary['Missing Count'] > 0])

             Missing Count  Missing %
Description           1454       0.27
CustomerID          135080      24.93


In [10]:
# Dropping rows where CustomerID is missing.
# CustomerID is the backbone of all customer-level analysis in this project —
# purchase frequency, revenue per customer, and repeat vs new customer logic
# all require a valid customer identifier. Rows without it cannot contribute
# to any of our four focus areas and must be removed.
# This affects ~24.9% of rows — a significant but justified removal.

df.dropna(subset=['CustomerID'], inplace=True)

print(f'Rows after dropping missing CustomerID: {len(df):,}')

Rows after dropping missing CustomerID: 406,829


In [11]:
# Dropping rows where Description is missing.
# Product descriptions are used in top-selling product analysis.
# Without a description, a product cannot be named or ranked meaningfully.
# Given that only 1,454 rows are affected (<0.3%), this is a safe removal
# with no material impact on dataset size or analysis quality.

df.dropna(subset=['Description'], inplace=True)

print(f'Rows after dropping missing Description: {len(df):,}')

Rows after dropping missing Description: 406,829


### 3.2 — Duplicate Removal

In [12]:
# Checking for fully duplicate rows — where every column value is identical.
# Duplicates in transaction data typically arise from system errors or
# double-logging events, and would artificially inflate revenue and frequency.

duplicate_count = df.duplicated().sum()
print(f'Duplicate rows found: {duplicate_count:,}')

Duplicate rows found: 5,225


In [13]:
# Removing duplicate rows, keeping the first occurrence.
# The first occurrence is retained as it represents the original transaction record.

df.drop_duplicates(inplace=True)

print(f'Rows after duplicate removal: {len(df):,}')

Rows after duplicate removal: 401,604


### 3.3 — Data Type Corrections

In [14]:
# Converting InvoiceDate from string (object) to datetime.
# This is essential for all time-based analysis in Week 3 —
# monthly revenue trends, peak purchasing periods, and seasonal patterns
# all require datetime operations that are impossible on string values.

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f'InvoiceDate dtype after conversion: {df["InvoiceDate"].dtype}')
print(f'Date range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')

InvoiceDate dtype after conversion: datetime64[ns]
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [15]:
# Converting CustomerID from float64 to string.
# CustomerID was loaded as a float because of NaN values (which pandas
# represents as float). Now that NaNs are removed, we convert to string
# to prevent accidental arithmetic on what is purely an identifier.
# We also strip the decimal point (e.g. 17850.0 → '17850') for clean display.

df['CustomerID'] = df['CustomerID'].astype(int).astype(str)

print(f'CustomerID dtype after conversion: {df["CustomerID"].dtype}')
print(f'Sample CustomerIDs: {df["CustomerID"].head(3).tolist()}')

CustomerID dtype after conversion: object
Sample CustomerIDs: ['17850', '17850', '17850']


### 3.4 — Removing Cancellations & Invalid Transactions

In [16]:
# Removing cancelled transactions — identified by InvoiceNo starting with 'C'.
# Cancellations represent reversed purchases and must be excluded because:
# 1. They would reduce revenue totals below actual earned revenue
# 2. They would distort purchase frequency counts per customer
# 3. They do not represent genuine customer purchasing behavior

cancelled_mask = df['InvoiceNo'].astype(str).str.startswith('C')
print(f'Cancelled transactions found: {cancelled_mask.sum():,}')

df = df[~cancelled_mask]
print(f'Rows after removing cancellations: {len(df):,}')

Cancelled transactions found: 8,872
Rows after removing cancellations: 392,732


In [17]:
# Removing rows where Quantity or UnitPrice is zero or negative.
# Using IQR logic to contextualize this decision:
# Negative quantities represent returns or data entry errors.
# Zero-price items represent samples or gifts — not revenue-generating transactions.
# Both categories would corrupt revenue calculations and spending distribution analysis.

before = len(df)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
after = len(df)

print(f'Rows removed due to invalid Quantity/UnitPrice: {before - after:,}')
print(f'Rows remaining: {after:,}')

Rows removed due to invalid Quantity/UnitPrice: 40
Rows remaining: 392,692


### 3.5 — Deriving the Revenue Column

In [18]:
# Creating a Revenue column as Quantity × UnitPrice.
# Revenue is our primary financial metric and does not exist as a raw column —
# it must be derived. This single column underpins:
# - Revenue per customer analysis
# - Top-selling product rankings
# - Country-level revenue comparison
# - Spending distribution analysis

df['Revenue'] = df['Quantity'] * df['UnitPrice']

print(f'Revenue column created.')
print(f'Total revenue in dataset: £{df["Revenue"].sum():,.2f}')
print(f'Average revenue per transaction: £{df["Revenue"].mean():,.2f}')
print(f'Median revenue per transaction: £{df["Revenue"].median():,.2f}')

Revenue column created.
Total revenue in dataset: £8,887,208.89
Average revenue per transaction: £22.63
Median revenue per transaction: £12.45


### 3.6 — Adding Useful Date Columns

In [19]:
# Extracting Month, Day of Week, and Hour from InvoiceDate.
# These derived columns avoid repeated datetime parsing in later sections
# and directly enable time-based visualizations in Week 3:
# - Month → monthly revenue trend line chart
# - DayOfWeek + Hour → purchase activity heatmap

df['Month'] = df['InvoiceDate'].dt.to_period('M').astype(str)
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour

print('Date columns extracted: Month, DayOfWeek, Hour')
print(f'Sample months: {df["Month"].unique()[:5].tolist()}')

Date columns extracted: Month, DayOfWeek, Hour
Sample months: ['2010-12', '2011-01', '2011-02', '2011-03', '2011-04']


### 3.7 — Final Dataset Summary

In [20]:
# Final inspection of the cleaned dataset before saving.
# Confirming: no missing values remain, all data types are correct,
# and the row count reflects all cleaning decisions made above.

print('=== CLEANED DATASET SUMMARY ===')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nMissing values remaining:')
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().any() else 'None — dataset is fully clean.')
print(f'\nData types:')
print(df.dtypes)
print(f'\nColumns: {df.columns.tolist()}')

=== CLEANED DATASET SUMMARY ===
Final shape: 392,692 rows × 12 columns

Missing values remaining:
None — dataset is fully clean.

Data types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
Revenue               float64
Month                  object
DayOfWeek              object
Hour                    int32
dtype: object

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Month', 'DayOfWeek', 'Hour']


### 3.8 — Saving the Cleaned Dataset

In [21]:
# Saving the cleaned dataset to the /data folder.
# index=False prevents pandas from writing the row index as an extra column,
# which would create an unnamed column that serves no analytical purpose.
# This file will be the single source of truth for all analysis in Weeks 3 and 4.

df.to_csv('/Users/mac/Desktop/DSA1080/Final Project/data/cleaned_dataset.csv', index=False)

print(f'Cleaned dataset saved to data/cleaned_dataset.csv')
print(f'Final row count: {len(df):,}')

Cleaned dataset saved to data/cleaned_dataset.csv
Final row count: 392,692
